# Exploring Qwen3.5 answers along with SMT

> Originally, adapted from [Qwen3_5_(0_8B)_Vision.ipynb](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Qwen3_5_(0_8B)_Vision.ipynb#scrollTo=gGFzmplrEy9I)

![Qwen3.5](https://qianwen-res.oss-accelerate.aliyuncs.com/logo_qwen3.5.png)

Qwen3.5 features the following enhancement:

- **Unified Vision-Language Foundation**: Early fusion training on multimodal tokens achieves cross-generational parity with Qwen3 and outperforms Qwen3-VL models across reasoning, coding, agents, and visual understanding benchmarks.
- **Efficient Hybrid Architecture**: Gated Delta Networks combined with sparse Mixture-of-Experts deliver high-throughput inference with minimal latency and cost overhead.
- **Scalable RL Generalization**: Reinforcement learning scaled across million-agent environments with progressively complex task distributions for robust real-world adaptability.
- **Global Linguistic Coverage**: Expanded support to 201 languages and dialects, enabling inclusive, worldwide deployment with nuanced cultural and regional understanding.
- **Next-Generation Training Infrastructure**: Near-100% multimodal training efficiency compared to text-only training and asynchronous RL frameworks supporting massive-scale agent scaffolds and environment orchestration.

In [ ]:
# Force unsloth to be on top

import json
from pathlib import Path
import random

In [ ]:
BASE_DIR = Path.cwd().parent
CATEGORY = "test"

# Input Files
INITIAL_STATE_PATH = BASE_DIR / f"submission_finetuning_{CATEGORY}_state.json"
SMT_FILE = BASE_DIR / f"smt_{CATEGORY}_state.json"

<a name="Data"></a>
### 🧪 Data Preparation

To convert our Sci-ImageMiner VQA data into the format required by Qwen2-VL (specifically for use with Unsloth), we need to restructure the data into a "messages" format.

The Qwen/Unsloth format expects a list of conversations where the image and the text prompt are bundled together in the user role, and the ground truth is in the assistant role, as follows:

```python
[
    { "role": "user",
    "content": [{"type": "text",  "text": Q}, {"type": "image", "image": image} ]
    },
    { "role": "assistant",
    "content": [{"type": "text",  "text": A} ]
    },
]
```

In [ ]:
with open(SMT_FILE, "r") as f:
    smt_data = json.load(f)

with open(INITIAL_STATE_PATH, "r") as f:
    initial_state = json.load(f)

<a name="Playground"></a>
### 🥼 Playground

In [ ]:
valid_candidates = []

for sample_id, sub_figs in initial_state.items():
    for sub_fig, questions in sub_figs.items():
        for q_obj in questions:
            # Check if this is a List question
            if q_obj.get("answer_type", "").lower() == "list":
                # Fetch corresponding SMT execution data
                s_entry = (
                    smt_data.get(sample_id, {})
                    .get(sub_fig, {})
                    .get(q_obj["question"], {})
                )

                code = s_entry.get("code", "")
                output = s_entry.get("output", "N/A")

                # Condition for "successful execution": Code exists, output exists, and no obvious errors
                if code and output != "N/A" and "error" not in output.lower():
                    valid_candidates.append(
                        {
                            "sample_id": sample_id,
                            "sub_fig": sub_fig,
                            "q_obj": q_obj,
                            "smt_entry": s_entry,
                        }
                    )

if not valid_candidates:
    raise ValueError(
        "Could not find any 'List' answer types with a successfully executed SMT code in the dataset."
    )

# Randomly select one candidate from the valid pool
selected_candidate = random.choice(valid_candidates)

target_sample_id = selected_candidate["sample_id"]
target_sub_fig = selected_candidate["sub_fig"]
target_q_obj = selected_candidate["q_obj"]
smt_entry = selected_candidate["smt_entry"]

print(f"Sample ID: {target_sample_id}")
print(f"Subfigure: {target_sub_fig}")
print(f"Question: {target_q_obj['question']}")
print(f"Answer: {target_q_obj['answer']}")
print("-" * 30)
print("-" * 30)
print("--- SMT OUTPUT ---")
print(smt_entry["output"])